# 02 — Data Cleaning and Preprocessing

## Objective

This notebook cleans and prepares the airline flight dataset for exploratory data analysis.

The cleaning process includes:

- Removing unnecessary columns
- Checking duplicate records
- Checking missing values
- Validating data types
- Standardizing categorical values
- Validating flight duration, days left, and price
- Checking logical consistency between source and destination
- Investigating potential outliers
- Performing final data-quality validation
- Saving the cleaned dataset


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [4]:
DATA_PATH = "../data/raw/airlines_flights_data.csv"

df = pd.read_csv(DATA_PATH)
df.head()

,index,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953
1,1,SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953
2,2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956
3,3,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955
4,4,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955


In [ ]:
print(f"Rows: {df.shape[0]},Columns: {df.shape[1]}")    

Rows: 300153,Columns: 12


In [7]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('(', '').str.replace(')', '')
df.columns

Index(['index', 'airline', 'flight', 'source_city', 'departure_time', 'stops',
       'arrival_time', 'destination_city', 'class', 'duration', 'days_left',
       'price'],
      dtype='object')

In [8]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300153 entries, 0 to 300152
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   index             300153 non-null  int64  
 1   airline           300153 non-null  object 
 2   flight            300153 non-null  object 
 3   source_city       300153 non-null  object 
 4   departure_time    300153 non-null  object 
 5   stops             300153 non-null  object 
 6   arrival_time      300153 non-null  object 
 7   destination_city  300153 non-null  object 
 8   class             300153 non-null  object 
 9   duration          300153 non-null  float64
 10  days_left         300153 non-null  int64  
 11  price             300153 non-null  int64  
dtypes: float64(1), int64(3), object(8)
memory usage: 27.5+ MB


In [10]:
#clean whitespace from string columns
for i in df.select_dtypes(include=['object']).columns:
    df[i] = df[i].astype(str).str.strip()

In [11]:
#Normalize na values
na_values = ['nan', 'NaN', 'NAN', 'NA', 'na', 'N/A', 'n/a', 'null', 'NULL', 'Null']
for i in df.columns:
    if df[i].dtype == 'object':
        df[i] = df[i].replace(to_replace=na_values, value=np.nan)

In [12]:
df["index"].head(10)

0    0
1    1
2    2
3    3
4    4
5    5
6    6
7    7
8    8
9    9
Name: index, dtype: int64

In [13]:
df.drop(columns=["index"], inplace=True)

### Removed `index`

The `index` column was removed because it only represented row identifiers and did not contain analytical information.

In [14]:
df.rename(columns={"class":"travel_class"}, inplace=True)

In [15]:
duplicate_count = df.duplicated().sum()

print(f"Duplicate rows: {duplicate_count:,}")

Duplicate rows: 0


In [16]:
missing_values = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (df.isnull().mean() * 100).round(2)
})

missing_values

,missing_count,missing_percentage
airline,0,0.0
flight,0,0.0
source_city,0,0.0
departure_time,0,0.0
stops,0,0.0
arrival_time,0,0.0
destination_city,0,0.0
travel_class,0,0.0
duration,0,0.0
days_left,0,0.0


In [17]:
df[["duration", "days_left", "price"]].describe()


,duration,days_left,price
count,300153.000000,300153.000000,300153.000000
mean,12.221021,26.004751,20889.660523
std,7.191997,13.561004,22697.767366
min,0.830000,1.000000,1105.000000
25%,6.830000,15.000000,4783.000000
50%,11.250000,26.000000,7425.000000
75%,16.170000,38.000000,42521.000000
max,49.830000,49.000000,123071.000000


In [ ]:
df["airline"].value_counts()

airline
Vistara      127859
Air_India     80892
Indigo        43120
GO_FIRST      23173
AirAsia       16098
SpiceJet       9011
Name: count, dtype: int64

In [ ]:
df["source_city"].value_counts()


In [ ]:
df["destination_city"].value_counts()

In [22]:
df["stops"].value_counts()

stops
one            250863
zero            36004
two_or_more     13286
Name: count, dtype: int64

In [23]:
stop_mapping = {
    "zero": 0,
    "one": 1,
    "two_or_more": 2
}

df["stops_numeric"] = df["stops"].map(stop_mapping)

In [24]:
df[["stops", "stops_numeric"]].drop_duplicates()

,stops,stops_numeric
0,zero,0
18,one,1
175,two_or_more,2
